In [4]:
import sys
from pathlib import Path

# Ajouter la racine du projet (le dossier parent de notebooks/) au chemin Python
RACINE = Path.cwd().parent
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))

import json
from typing import Literal
from langchain_core.tools import tool

from src.recherche import collection, rechercher

In [6]:
print("Éléments dans la collection :", collection.count())

Éléments dans la collection : 166


In [12]:
def recuperer_sources(question, k=6, type_resultat=None):
    resultats = rechercher(texte=question, type_resultat=type_resultat, k=k)
    sources = []
    for i, r in enumerate(resultats, start=1):
        source = {"reference": f"S{i}"}
        source.update(r)
        sources.append(source)
    return sources

In [13]:
@tool
def consulter_pdf(
    question: str,
    type_resultat: Literal["tous", "texte", "image"] = "tous",
    k: int = 6,
) -> str:
    """
    Rechercher dans le PDF indexé (article "Attention Is All You Need")
    les passages de texte et les figures liés à la question.

    Utiliser cet outil pour toute question sur le contenu du document :
    explications, formules, méthodes, résultats, figures.

    type_resultat :
    - "tous"  : textes et images ;
    - "texte" : uniquement des passages textuels ;
    - "image" : uniquement des figures.

    Renvoie un JSON avec les sources trouvées (référence, page, similarité,
    contenu pour le texte, chemin pour les images).
    """
    filtre = None if type_resultat == "tous" else type_resultat
    k = max(1, min(k, 12))

    try:
        sources = recuperer_sources(question, k=k, type_resultat=filtre)
    except Exception as erreur:
        return json.dumps({"erreur": str(erreur), "sources": []}, ensure_ascii=False)

    return json.dumps(
        {"question": question, "sources": sources},
        ensure_ascii=False,
    )

In [14]:
print("Nom        :", consulter_pdf.name)
print("Arguments  :", consulter_pdf.args)

sortie = consulter_pdf.invoke({
    "question": "Qu'est-ce que le mécanisme d'attention multi-têtes ?",
    "type_resultat": "texte",
    "k": 3,
})
print(json.dumps(json.loads(sortie), indent=2, ensure_ascii=False))

sortie_img = consulter_pdf.invoke({
    "question": "architecture du Transformer",
    "type_resultat": "image",
    "k": 2,
})
print(json.dumps(json.loads(sortie_img), indent=2, ensure_ascii=False))

Nom        : consulter_pdf
Arguments  : {'question': {'title': 'Question', 'type': 'string'}, 'type_resultat': {'default': 'tous', 'enum': ['tous', 'texte', 'image'], 'title': 'Type Resultat', 'type': 'string'}, 'k': {'default': 6, 'title': 'K', 'type': 'integer'}}
{
  "question": "Qu'est-ce que le mécanisme d'attention multi-têtes ?",
  "sources": [
    {
      "reference": "S1",
      "id": "bdfaa68d8984f0dc02beaca5:texte:2:8",
      "type": "texte",
      "page": 2,
      "contenu": "reduced effective resolution due to averaging attention - weighted positions , an effect we counteract with multi - head attention as described in section 3 . 2 . self - attention , sometimes called intra - attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence . self - attention has been used successfully in a variety of tasks including reading",
      "chemin": null,
      "similarite": 0.59
    },
    {
      "referen

In [11]:
r = rechercher(texte="multi-head attention", k=2)
print(type(r))
print(r)

<class 'list'>
[{'id': 'bdfaa68d8984f0dc02beaca5:texte:14:2', 'type': 'texte', 'page': 14, 'contenu': ", also in layer 5 of 6 , apparently involved in anaphora resolution . top : full attentions for head 5 . bottom : isolated attentions from just the word ' its ' for attention heads 5 and 6 . note that the attentions are very sharp for this word . 1 4", 'chemin': None, 'similarite': 0.669}, {'id': 'bdfaa68d8984f0dc02beaca5:texte:2:8', 'type': 'texte', 'page': 2, 'contenu': 'reduced effective resolution due to averaging attention - weighted positions , an effect we counteract with multi - head attention as described in section 3 . 2 . self - attention , sometimes called intra - attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence . self - attention has been used successfully in a variety of tasks including reading', 'chemin': None, 'similarite': 0.654}]
